# Riesgo de crédito en un banco
## Notebook 02 · Construcción de variables con SQL

**Entrada:** `data/banco.db`, generada en `01_exploracion.ipynb`.

**Qué hace este notebook:** para cada préstamo, resume el comportamiento de la cuenta antes de la fecha del préstamo: cuántas transacciones hizo, su saldo promedio y mínimo, cuántas veces quedó en negativo y cuántos intereses sancionatorios pagó.

**Regla principal (anti-fuga de información):** solo se usan transacciones con fecha **anterior** a la del préstamo. El banco, al aprobar el préstamo, no conocía lo que pasaría después.

**Resultado:** tabla `prestamos_variables` en la base de datos y `data/processed/prestamos_variables.csv`.

In [1]:
import os
import sqlite3
import pandas as pd

conexion = sqlite3.connect("../data/banco.db")

def consulta(sql):
    """Ejecuta una consulta SQL y devuelve el resultado como DataFrame."""
    return pd.read_sql(sql, conexion)

consulta("SELECT COUNT(*) AS prestamos FROM loan")

,prestamos
0,682


## 1. Historial de cada préstamo

Primero se unen los préstamos con las transacciones de su cuenta, pero solo las anteriores a la fecha del préstamo. La condición de fecha va dentro del `ON` del `JOIN`.

In [18]:
# Primero confirmo el nombre de las columnas de prestamos (loan.csv) y transacciones (trans.csv)
print("Columnas de prestamos: ")
consulta("PRAGMA table_info('loan')")



Columnas de prestamos: 


,cid,name,type,notnull,dflt_value,pk
0,0,loan_id,INTEGER,0,None,0
1,1,account_id,INTEGER,0,None,0
2,2,date,TEXT,0,None,0
3,3,amount,INTEGER,0,None,0
4,4,duration,INTEGER,0,None,0
5,5,payments,REAL,0,None,0
6,6,status,TEXT,0,None,0
7,7,malo,INTEGER,0,None,0


In [17]:
print("Columnas de transacciones: ")
consulta("PRAGMA table_info('trans')")

Columnas de transacciones: 


,cid,name,type,notnull,dflt_value,pk
0,0,trans_id,INTEGER,0,None,0
1,1,account_id,INTEGER,0,None,0
2,2,date,TEXT,0,None,0
3,3,type,TEXT,0,None,0
4,4,operation,TEXT,0,None,0
5,5,amount,INTEGER,0,None,0
6,6,balance,INTEGER,0,None,0
7,7,k_symbol,TEXT,0,None,0
8,8,bank,TEXT,0,None,0
9,9,account,REAL,0,None,0


Identifico por cual fila conectar `account_id` y hago el JOIN LEFT para poder quedarme con la info de transacciones y lo que necesite de prestamos.

In [19]:
consulta("""
SELECT
    l.loan_id,
    l.date              AS fecha_prestamo,
    COUNT(t.trans_id)   AS n_transacciones,
    MIN(t.date)         AS primera_transaccion,
    MAX(t.date)         AS ultima_transaccion
FROM loan AS l
LEFT JOIN trans AS t
    ON t.account_id = l.account_id
    AND t.date < l.date
GROUP BY l.loan_id, l.date
ORDER BY n_transacciones
LIMIT 10
""")

,loan_id,fecha_prestamo,n_transacciones,primera_transaccion,ultima_transaccion
0,5063,1997-07-10,2,1997-03-30,1997-06-28
1,5314,1993-07-05,4,1993-03-22,1993-06-20
2,6594,1994-09-28,5,1994-05-17,1994-09-14
3,5253,1997-06-03,10,1997-02-20,1997-05-31
4,5301,1996-07-06,10,1996-03-14,1996-06-30
5,5892,1996-05-14,10,1995-12-27,1996-04-30
6,6736,1994-06-14,10,1994-02-27,1994-05-31
7,5765,1998-02-10,11,1997-10-18,1998-02-08
8,5973,1997-11-24,11,1997-08-08,1997-11-10
9,6354,1996-10-23,11,1996-06-25,1996-10-12
